In [1]:
import sqlite3
import pandas as pd

# Load clean data
clean_df = pd.read_csv(r'D:\plfs_project\data\plfs_clean.csv')

# Create SQLite database
conn = sqlite3.connect(r'D:\plfs_project\data\plfs_database.db')

# Push into SQLite
clean_df.to_sql('plfs', conn, if_exists='replace', index=False)

print("✅ Database created!")
print(f"   Rows loaded: {len(clean_df):,}")
print(f"   Columns    : {list(clean_df.columns)}")

✅ Database created!
   Rows loaded: 277,485
   Columns    : ['ST', 'state_name', 'DC', 'STRM', 'area_type', 'SEX', 'gender', 'AGE', 'GEDU_LVL', 'education', 'PAS', 'activity_status', 'lf_status', 'ERN_REG', 'ERN_SELF', 'MULT', 'total_earnings']


In [2]:
def run_query(sql, title=""):
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    print(result.to_string(index=False))
    return result

print("✅ Helper function ready! Starting SQL queries...")

✅ Helper function ready! Starting SQL queries...


In [3]:
run_query("""
    SELECT 
        state_name,
        COUNT(*) AS total_people,
        SUM(CASE WHEN lf_status = 'Employed' THEN 1 ELSE 0 END) AS employed,
        SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) AS unemployed,
        ROUND(
            SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) * 100.0 /
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END), 1
        ) AS unemployment_rate_pct
    FROM plfs
    WHERE state_name IS NOT NULL
    GROUP BY state_name
    ORDER BY unemployment_rate_pct DESC
""", "Q1 — Unemployment Rate by State (Highest First)")


  Q1 — Unemployment Rate by State (Highest First)
                  state_name  total_people  employed  unemployed  unemployment_rate_pct
               A & N Islands          1198       692         114                   14.1
                    Nagaland          2786      1778         225                   11.2
                   Meghalaya          4146      2514         310                   11.0
                 Lakshadweep           494       239          26                    9.8
                      Kerala          9387      4675         497                    9.6
                  Chandigarh           980       560          58                    9.4
           Arunachal Pradesh          5300      3366         344                    9.3
            Jammu  & Kashmir          9276      4613         458                    9.0
                         Goa          1063       595          58                    8.9
                      Ladakh           682       346          33     

,state_name,total_people,employed,unemployed,unemployment_rate_pct
0,A & N Islands,1198,692,114,14.1
1,Nagaland,2786,1778,225,11.2
2,Meghalaya,4146,2514,310,11.0
3,Lakshadweep,494,239,26,9.8
4,Kerala,9387,4675,497,9.6
5,Chandigarh,980,560,58,9.4
6,Arunachal Pradesh,5300,3366,344,9.3
7,Jammu & Kashmir,9276,4613,458,9.0
8,Goa,1063,595,58,8.9
9,Ladakh,682,346,33,8.7


In [4]:
run_query("""
    SELECT 
        education,
        COUNT(*) AS total_in_labour_force,
        SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) AS unemployed,
        ROUND(
            SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) * 100.0 /
            COUNT(*), 1
        ) AS unemployment_rate_pct
    FROM plfs
    WHERE lf_status IN ('Employed', 'Unemployed')
    AND education IS NOT NULL
    GROUP BY education
    ORDER BY unemployment_rate_pct DESC
""", "Q2 — Unemployment Rate by Education Level")


  Q2 — Unemployment Rate by Education Level
              education  total_in_labour_force  unemployed  unemployment_rate_pct
               Graduate                  22349        3758                   16.8
   Postgraduate & Above                   7052        1107                   15.7
                Diploma                   3376         414                   12.3
       Higher Secondary                  19233        1297                    6.7
              Secondary                  22047         737                    3.3
Literate (No Schooling)                    179           5                    2.8
                 Middle                  36651         990                    2.7
                Primary                  19987         249                    1.2
          Below Primary                   7774          44                    0.6
           Not Literate                  22770          91                    0.4


,education,total_in_labour_force,unemployed,unemployment_rate_pct
0,Graduate,22349,3758,16.8
1,Postgraduate & Above,7052,1107,15.7
2,Diploma,3376,414,12.3
3,Higher Secondary,19233,1297,6.7
4,Secondary,22047,737,3.3
5,Literate (No Schooling),179,5,2.8
6,Middle,36651,990,2.7
7,Primary,19987,249,1.2
8,Below Primary,7774,44,0.6
9,Not Literate,22770,91,0.4


In [5]:
run_query("""
    SELECT
        gender,
        area_type,
        COUNT(*) AS total,
        ROUND(
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END) * 100.0 /
            COUNT(*), 1
        ) AS lfpr_pct,
        ROUND(
            SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) * 100.0 /
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END), 1
        ) AS unemployment_rate_pct
    FROM plfs
    WHERE gender IN ('Male','Female')
    AND area_type IS NOT NULL
    GROUP BY gender, area_type
    ORDER BY gender, area_type
""", "Q3 — LFPR and Unemployment by Gender and Area")


  Q3 — LFPR and Unemployment by Gender and Area
gender area_type  total  lfpr_pct  unemployment_rate_pct
Female     Rural 129466      35.4                    7.1
Female     Urban   9562      31.3                    2.6
  Male     Rural 128543      81.3                    4.9
  Male     Urban   9901      82.4                    2.9


,gender,area_type,total,lfpr_pct,unemployment_rate_pct
0,Female,Rural,129466,35.4,7.1
1,Female,Urban,9562,31.3,2.6
2,Male,Rural,128543,81.3,4.9
3,Male,Urban,9901,82.4,2.9


In [6]:
run_query("""
    SELECT
        CASE 
            WHEN AGE BETWEEN 15 AND 24 THEN '15-24 (Youth)'
            WHEN AGE BETWEEN 25 AND 34 THEN '25-34 (Young Adult)'
            WHEN AGE BETWEEN 35 AND 44 THEN '35-44 (Mid Career)'
            WHEN AGE BETWEEN 45 AND 60 THEN '45-60 (Experienced)'
        END AS age_group,
        COUNT(*) AS total,
        ROUND(
            SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) * 100.0 /
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END), 1
        ) AS unemployment_rate_pct,
        ROUND(
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END) * 100.0 /
            COUNT(*), 1
        ) AS lfpr_pct
    FROM plfs
    WHERE AGE BETWEEN 15 AND 60
    GROUP BY age_group
    ORDER BY unemployment_rate_pct DESC
""", "Q4 — Unemployment by Age Group")


  Q4 — Unemployment by Age Group
          age_group  total  unemployment_rate_pct  lfpr_pct
      15-24 (Youth)  75145                   20.1      29.9
25-34 (Young Adult)  64509                    8.1      67.1
 35-44 (Mid Career)  62170                    1.1      71.6
45-60 (Experienced)  75661                    0.3      67.6


,age_group,total,unemployment_rate_pct,lfpr_pct
0,15-24 (Youth),75145,20.1,29.9
1,25-34 (Young Adult),64509,8.1,67.1
2,35-44 (Mid Career),62170,1.1,71.6
3,45-60 (Experienced),75661,0.3,67.6


In [7]:
run_query("""
    SELECT
        activity_status,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM plfs
    WHERE lf_status = 'Employed'
    AND activity_status IS NOT NULL
    GROUP BY activity_status
    ORDER BY count DESC
""", "Q5 — How Are People Employed? (Type of Employment)")


  Q5 — How Are People Employed? (Type of Employment)
             activity_status  count  percentage
 Self-Employed (Own Account)  54206        35.5
       Regular Salaried/Wage  45932        30.1
       Casual Labour (Other)  26896        17.6
        Unpaid Family Worker  19644        12.9
    Self-Employed (Employer)   5457         3.6
Casual Labour (Public Works)    591         0.4


,activity_status,count,percentage
0,Self-Employed (Own Account),54206,35.5
1,Regular Salaried/Wage,45932,30.1
2,Casual Labour (Other),26896,17.6
3,Unpaid Family Worker,19644,12.9
4,Self-Employed (Employer),5457,3.6
5,Casual Labour (Public Works),591,0.4


In [8]:
run_query("""
    SELECT
        state_name,
        gender,
        ROUND(
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END) * 100.0 /
            COUNT(*), 1
        ) AS lfpr_pct
    FROM plfs
    WHERE gender IN ('Male','Female')
    AND state_name IS NOT NULL
    GROUP BY state_name, gender
    HAVING gender = 'Female'
    ORDER BY lfpr_pct ASC
    LIMIT 5
""", "Q6 — Bottom 5 States for Female LFPR (Worst Gender Gap)")


  Q6 — Bottom 5 States for Female LFPR (Worst Gender Gap)
    state_name gender  lfpr_pct
         Bihar Female      13.8
         Delhi Female      16.4
 Uttar Pradesh Female      17.3
   Lakshadweep Female      21.3
Madhya Pradesh Female      22.0


,state_name,gender,lfpr_pct
0,Bihar,Female,13.8
1,Delhi,Female,16.4
2,Uttar Pradesh,Female,17.3
3,Lakshadweep,Female,21.3
4,Madhya Pradesh,Female,22.0


In [13]:
run_query("""
    SELECT
        education,
        gender,
        ROUND(AVG(CASE WHEN total_earnings > 0 THEN total_earnings END), 0) AS avg_monthly_earnings,
        COUNT(CASE WHEN total_earnings > 0 THEN 1 END) AS earners
    FROM plfs
    WHERE lf_status = 'Employed'
    AND education IS NOT NULL
    AND gender IN ('Male','Female')
    GROUP BY education, gender
    ORDER BY 
        CASE education
            WHEN 'Postgraduate & Above' THEN 1
            WHEN 'Graduate' THEN 2
            WHEN 'Diploma' THEN 3
            WHEN 'Higher Secondary' THEN 4
            WHEN 'Secondary' THEN 5
            WHEN 'Middle' THEN 6
            WHEN 'Primary' THEN 7
            WHEN 'Below Primary' THEN 8
            WHEN 'Literate (No Schooling)' THEN 9
            WHEN 'Not Literate' THEN 10
        END,
        CASE WHEN gender = 'Male' THEN 1 ELSE 2 END
""", "Q7 — Average Monthly Earnings by Education and Gender")


  Q7 — Average Monthly Earnings by Education and Gender
              education gender  avg_monthly_earnings  earners
   Postgraduate & Above   Male               42540.0     3682
   Postgraduate & Above Female               33537.0     1885
               Graduate   Male               30988.0    12595
               Graduate Female               24147.0     3690
                Diploma   Male               25536.0     2094
                Diploma Female               19834.0      451
       Higher Secondary   Male               20412.0    10887
       Higher Secondary Female               11276.0     2542
              Secondary   Male               18314.0    12115
              Secondary Female                8885.0     2732
                 Middle   Male               16364.0    17972
                 Middle Female                7987.0     5289
                Primary   Male               15248.0     8451
                Primary Female                6908.0     3412
          Bel

,education,gender,avg_monthly_earnings,earners
0,Postgraduate & Above,Male,42540.0,3682
1,Postgraduate & Above,Female,33537.0,1885
2,Graduate,Male,30988.0,12595
3,Graduate,Female,24147.0,3690
4,Diploma,Male,25536.0,2094
5,Diploma,Female,19834.0,451
6,Higher Secondary,Male,20412.0,10887
7,Higher Secondary,Female,11276.0,2542
8,Secondary,Male,18314.0,12115
9,Secondary,Female,8885.0,2732


In [14]:
run_query("""
    SELECT
        area_type,
        education,
        COUNT(*) AS total,
        ROUND(
            SUM(CASE WHEN lf_status = 'Unemployed' THEN 1 ELSE 0 END) * 100.0 /
            SUM(CASE WHEN lf_status IN ('Employed','Unemployed') THEN 1 ELSE 0 END), 1
        ) AS unemployment_rate_pct
    FROM plfs
    WHERE education IN ('Graduate','Postgraduate & Above','Higher Secondary')
    AND area_type IS NOT NULL
    GROUP BY area_type, education
    ORDER BY unemployment_rate_pct DESC
""", "Q8 — Educated Unemployment: Rural vs Urban Breakdown")


  Q8 — Educated Unemployment: Rural vs Urban Breakdown
area_type            education  total  unemployment_rate_pct
    Rural             Graduate  30530                   17.3
    Rural Postgraduate & Above   8745                   16.1
    Urban Postgraduate & Above    550                    9.5
    Urban             Graduate   1905                    9.2
    Rural     Higher Secondary  38936                    7.0
    Urban     Higher Secondary   2900                    3.9


,area_type,education,total,unemployment_rate_pct
0,Rural,Graduate,30530,17.3
1,Rural,Postgraduate & Above,8745,16.1
2,Urban,Postgraduate & Above,550,9.5
3,Urban,Graduate,1905,9.2
4,Rural,Higher Secondary,38936,7.0
5,Urban,Higher Secondary,2900,3.9


In [15]:
# Check exact unique values in lf_status column
print(clean_df['lf_status'].unique())
print()
print(clean_df['lf_status'].value_counts())


['Employed' 'Not in Labour Force' 'Unemployed']

lf_status
Employed               152726
Not in Labour Force    116067
Unemployed               8692
Name: count, dtype: int64
